# Plant-Disease Model — Architecture, Evaluation, and Inference Reference

This notebook documents the **deployed** Smart Harvest AI plant-disease classifier. It does not retrain or modify the model. It explains the training architecture, inspects the TensorFlow Lite artifact, displays the measured PlantVillage validation metrics, visualizes the confusion matrix, and demonstrates inference on one image.

**Deployed model:** `models/disease_model.tflite`  
**Labels:** `models/class_labels.json`  
**Evaluation report:** `models/disease_metrics.json`  
**Training source:** `backend/ml/train_disease_model.py`  
**Reproducible evaluator:** `backend/ml/evaluate_disease_model.py`


## 1. Model design

The training pipeline uses transfer learning with:

- **Backbone:** ImageNet-pretrained `MobileNetV3Small`
- **Input:** 192 × 192 RGB image
- **Backbone preprocessing:** disabled; input pixels are explicitly scaled to `[0, 1]`
- **Classifier head:** global average pooling → dense layer with 128 ReLU units → dropout 0.3 → 15-class softmax
- **Loss:** sparse categorical cross-entropy
- **Optimizer:** Adam
- **Training strategy:** frozen backbone followed by fine-tuning of the final 40 backbone layers
- **Dataset split:** seeded 80/20 training-validation split with seed 123
- **Deployment:** TensorFlow Lite

The `.tflite` format is optimized for inference and does not preserve a Keras-style layer summary. The architecture description above comes from the reproducible training script; the notebook separately inspects the deployed model's input and output tensors.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

# Support launching Jupyter from either the repository root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "models").exists():
    ROOT = ROOT.parent

MODEL_PATH = ROOT / "models" / "disease_model.tflite"
LABELS_PATH = ROOT / "models" / "class_labels.json"
METRICS_PATH = ROOT / "models" / "disease_metrics.json"
CONFUSION_PATH = ROOT / "models" / "disease_confusion_matrix.png"
DATASET_DIR = ROOT / "PlantVillage" / "PlantVillage"
IMAGE_SIZE = (192, 192)

for required_path in (MODEL_PATH, LABELS_PATH, METRICS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required artifact not found: {required_path}")

print(f"Repository root: {ROOT.resolve()}")
print(f"TFLite model size: {MODEL_PATH.stat().st_size / 1024:.1f} KiB")


## 2. Class labels

The class index order must exactly match the model's 15 softmax outputs.


In [ ]:
labels_map = json.loads(LABELS_PATH.read_text())
class_names = [labels_map[str(index)] for index in range(len(labels_map))]
labels_table = pd.DataFrame({"class_index": range(len(class_names)), "class_name": class_names})
labels_table


## 3. Inspect the deployed TensorFlow Lite model

This verifies the deployed input/output contract used by the Flask application.


In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

model_contract = pd.DataFrame([
    {
        "tensor": "input",
        "name": input_detail["name"],
        "shape": tuple(int(value) for value in input_detail["shape"]),
        "dtype": np.dtype(input_detail["dtype"]).name,
        "quantization": input_detail["quantization"],
    },
    {
        "tensor": "output",
        "name": output_detail["name"],
        "shape": tuple(int(value) for value in output_detail["shape"]),
        "dtype": np.dtype(output_detail["dtype"]).name,
        "quantization": output_detail["quantization"],
    },
])
model_contract


In [ ]:
assert tuple(input_detail["shape"]) == (1, 192, 192, 3), "Unexpected model input shape"
assert int(output_detail["shape"][-1]) == len(class_names), "Output count and labels differ"
print(f"Verified: one {IMAGE_SIZE[0]}×{IMAGE_SIZE[1]} RGB image produces {len(class_names)} class scores.")


## 4. Verified validation metrics

The saved report was generated by evaluating the deployed TFLite artifact on the seeded 20% PlantVillage validation split. Weighted metrics account for class support; macro metrics give every class equal weight.


In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
summary = pd.DataFrame([
    {"metric": "Accuracy", "score": metrics["accuracy"]},
    {"metric": "Weighted precision", "score": metrics["weighted_precision"]},
    {"metric": "Weighted recall", "score": metrics["weighted_recall"]},
    {"metric": "Weighted F1", "score": metrics["weighted_f1"]},
    {"metric": "Macro precision", "score": metrics["macro_precision"]},
    {"metric": "Macro recall", "score": metrics["macro_recall"]},
    {"metric": "Macro F1", "score": metrics["macro_f1"]},
])
summary["percentage"] = summary["score"].map(lambda value: f"{value * 100:.2f}%")
print(f"Validation images: {metrics['validation_images']:,}")
print(f"Classes: {metrics['classes']}")
print(f"Validation split: {metrics['validation_split']:.0%}, seed {metrics['split_seed']}")
summary[["metric", "percentage"]]


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(summary["metric"], summary["score"] * 100, color="#3d7a45")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 100)
ax.set_title("Deployed Disease Model — PlantVillage Validation Metrics")
ax.tick_params(axis="x", rotation=35)
ax.bar_label(bars, fmt="%.2f%%", padding=3)
plt.tight_layout()
plt.show()


## 5. Per-class precision, recall, F1, and support


In [ ]:
per_class = (
    pd.DataFrame.from_dict(metrics["per_class"], orient="index")
    .rename_axis("class_name")
    .reset_index()
)
per_class[["precision", "recall", "f1"]] *= 100
per_class.sort_values("f1", ascending=False).style.format({
    "precision": "{:.2f}%",
    "recall": "{:.2f}%",
    "f1": "{:.2f}%",
})


In [ ]:
plot_data = per_class.sort_values("f1")
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(plot_data["class_name"], plot_data["f1"], color="#6ca678")
ax.set_xlabel("F1 score (%)")
ax.set_xlim(0, 100)
ax.set_title("Per-Class F1 Scores")
plt.tight_layout()
plt.show()


## 6. Confusion matrix

Rows are true classes and columns are predicted classes. The committed PNG is shown first; the second cell can recreate the plot from raw values in the JSON report.


In [ ]:
if CONFUSION_PATH.exists():
    matrix_image = Image.open(CONFUSION_PATH)
    display(matrix_image)
else:
    print(f"Rendered confusion matrix not found: {CONFUSION_PATH}")


In [ ]:
matrix = np.asarray(metrics["confusion_matrix"], dtype=int)
assert matrix.shape == (len(class_names), len(class_names))

fig, ax = plt.subplots(figsize=(14, 12))
image = ax.imshow(matrix, cmap="Greens")
ax.set(
    xticks=range(len(class_names)),
    yticks=range(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted class",
    ylabel="True class",
    title="Disease Model Confusion Matrix",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 7. Run inference on one image

Set `IMAGE_PATH` to any pepper, potato, or tomato leaf image. This preprocessing matches the Flask disease endpoint: RGB conversion, resize to 192 × 192, float32 conversion, and division by 255. The cell reports the top three softmax scores.


In [ ]:
def predict_leaf(image_path, top_k=3):
    image = Image.open(image_path).convert("RGB").resize(IMAGE_SIZE)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = np.expand_dims(array, axis=0)

    if input_detail["dtype"] != np.float32:
        scale, zero_point = input_detail["quantization"]
        array = np.round(array / scale + zero_point).astype(input_detail["dtype"])

    interpreter.set_tensor(input_detail["index"], array)
    interpreter.invoke()
    scores = interpreter.get_tensor(output_detail["index"])[0]

    if output_detail["dtype"] != np.float32:
        scale, zero_point = output_detail["quantization"]
        scores = (scores.astype(np.float32) - zero_point) * scale

    top_indices = np.argsort(scores)[-top_k:][::-1]
    return pd.DataFrame({
        "class_name": [class_names[index] for index in top_indices],
        "confidence": [float(scores[index]) for index in top_indices],
    })

# Example:
# IMAGE_PATH = DATASET_DIR / "Tomato_healthy" / "example.jpg"
# predict_leaf(IMAGE_PATH).style.format({"confidence": "{:.2%}"})


## 8. Reproduce the complete evaluation

From the repository root, run:

```bash
python backend/ml/evaluate_disease_model.py
```

This requires the local dataset at `PlantVillage/PlantVillage/`. It evaluates the existing model—without retraining—and rewrites:

- `models/disease_metrics.json`
- `models/disease_confusion_matrix.png`

The split configuration must remain at validation fraction `0.2` and seed `123` to remain comparable with the training pipeline.
